# ЛР №4. Разведочный анализ данных Audi

## Загрузка данных и первичный анализ

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import RobustScaler, StandardScaler

raw = pd.read_csv(Path("data") / "audi.csv")
target = "price"
categories = ["model", "transmission", "fuelType"]
print(f"Строк: {len(raw):,}. Входных признаков: {raw.shape[1] - 1}.")
display(pd.DataFrame({
    "Тип хранения": raw.dtypes.astype(str),
    "Пропуски": raw.isna().sum(),
    "Уникальные значения": raw.nunique(),
}))
display(raw.describe().T.round(2))
for column in categories:
    counts = raw[column].str.strip().value_counts()
    display(pd.DataFrame({"Частота": counts, "Доля, %": (100 * counts / len(raw)).round(2)}))
print(f"Строки, повторяющие предыдущие: {raw.duplicated().sum()}")
print(f"Нулевой объём двигателя: {raw['engineSize'].eq(0).sum()}")
print(f"Нулевой налог: {raw['tax'].eq(0).sum()}")

## Подготовка аналитической выборки

In [ ]:
data = raw.loc[raw["engineSize"].gt(0)].copy()
data[categories] = data[categories].apply(lambda column: column.str.strip())
print(f"Строк в аналитической выборке: {len(data):,}")

## Кодирование категориальных признаков

In [ ]:
encoded = pd.get_dummies(data[categories], dtype=int)
display(encoded.head())
print(f"Двоичных признаков: {encoded.shape[1]}")

## Конструирование признаков

In [ ]:
median_mileage = data.groupby("year")["mileage"].transform("median")
data["mileage_vs_year_median"] = data["mileage"] - median_mileage
data["fuel_per_engine_liter"] = (282.481 / data["mpg"]) / data["engineSize"]
new_features = ["mileage_vs_year_median", "fuel_per_engine_liter"]
display(data[new_features].describe().T.round(2))

## Построение диаграмм рассеяния

In [ ]:
quantitative = ["mileage", "tax", "mpg", "engineSize"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, column in zip(axes.flat, quantitative):
    ax.scatter(data[column], data[target], s=5, alpha=0.16)
    ax.set(xlabel=column, ylabel="Цена, £")
plt.show()

## Линеаризация и сравнение зависимостей

In [ ]:
transforms = {
    "mpg": "fuel_l_per_100km",
    "engineSize": "engine_size_squared",
}
data["fuel_l_per_100km"] = 282.481 / data["mpg"]
data["engine_size_squared"] = data["engineSize"].pow(2)

linearity = []
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for row, (original, transformed) in enumerate(transforms.items()):
    for ax, column in zip(axes[row], (original, transformed)):
        x = data[column]
        y = data[target]
        slope, intercept = np.polyfit(x, y, 1)
        domain = np.linspace(x.min(), x.max(), 100)
        ax.scatter(x, y, s=5, alpha=0.16)
        ax.plot(domain, slope * domain + intercept, color="crimson")
        ax.set(xlabel=column, ylabel="Цена, £")
        pearson = x.corr(y)
        linearity.append((original, column, pearson, pearson ** 2))
plt.show()
display(pd.DataFrame(
    linearity, columns=["Исходный признак", "На графике", "Пирсон", "R² линейной аппроксимации"]
).round(4))

## Расчёт и визуализация корреляций Пирсона и Спирмена

In [ ]:
original_numeric = ["year", "price", *quantitative]
analysis = pd.concat(
    [data[original_numeric + new_features + list(transforms.values())], encoded],
    axis=1,
)
pearson = analysis.corr(method="pearson")
spearman = analysis.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(24, 12), constrained_layout=True)
for ax, name, matrix in zip(axes, ["Пирсон", "Спирмен"], [pearson, spearman]):
    image = ax.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
    ticks = range(len(matrix))
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(matrix.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(matrix.index, fontsize=7)
    ax.set_title(name)
fig.colorbar(image, ax=axes, shrink=0.6)
plt.show()

## Сопоставление корреляций

In [ ]:
price_correlations = pd.DataFrame({
    "Пирсон": pearson[target],
    "Спирмен": spearman[target],
}).drop(target)
price_correlations["Модуль разности"] = (
    price_correlations["Пирсон"] - price_correlations["Спирмен"]
).abs()
display(price_correlations.round(4))

rows, cols = np.triu_indices(len(pearson), k=1)
pairs = pd.DataFrame({
    "Признак 1": pearson.index[rows],
    "Признак 2": pearson.columns[cols],
    "Пирсон": pearson.to_numpy()[rows, cols],
    "Спирмен": spearman.to_numpy()[rows, cols],
})
pairs["Модуль разности"] = (pairs["Пирсон"] - pairs["Спирмен"]).abs()
display(pairs.nlargest(10, "Модуль разности").round(4))

## Сравнение способов масштабирования

In [ ]:
scale_sample = ["mileage", "engineSize"]
variants = {"Исходные": data[scale_sample]}
for scaler in (StandardScaler(), RobustScaler()):
    variants[type(scaler).__name__] = pd.DataFrame(
        scaler.fit_transform(data[scale_sample]), columns=scale_sample, index=data.index
    )

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
for col_idx, (name, values) in enumerate(variants.items()):
    for row_idx, column in enumerate(scale_sample):
        ax = axes[row_idx, col_idx]
        values[column].plot.kde(ax=ax)
        ax.set(xlabel=column, ylabel="Плотность", title=name)
plt.show()
display(pd.concat({
    name: values.describe().T.assign(IQR=values.quantile(0.75) - values.quantile(0.25))
    for name, values in variants.items()
}).round(4))

## Применение выбранного масштабирования

In [ ]:
scale_columns = [*quantitative, *new_features, *transforms.values()]
scaled = analysis.copy()
scaled[scale_columns] = RobustScaler().fit_transform(analysis[scale_columns])
display(scaled.head())
print(f"Итоговый размер таблицы: {scaled.shape}")